In [3]:
import pandas as pd

df = pd.read_csv("vendas.csv", sep=";")

In [4]:
# Exibir informações gerais sobre o DataFrame
print("--- Informações do DataFrame ---")
display(df.info())

print("\n--- Resumo Estatístico de Variáveis Numéricas ---")
display(df.describe())

print("\n--- Primeiras 5 linhas do DataFrame ---")
display(df.head())

--- Informações do DataFrame ---
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_item_venda      50000 non-null  int64  
 1   id_pedido          50000 non-null  int64  
 2   data_venda         50000 non-null  str    
 3   nome_produto       50000 non-null  str    
 4   categoria_produto  50000 non-null  str    
 5   quantidade         50000 non-null  int64  
 6   preco_unitario     50000 non-null  float64
 7   desconto           50000 non-null  float64
 8   valor_total        50000 non-null  float64
 9   nome_cliente       50000 non-null  str    
 10  cidade_cliente     49850 non-null  str    
 11  estado_cliente     50000 non-null  str    
 12  nome_vendedor      50000 non-null  str    
 13  forma_pagamento    50000 non-null  str    
 14  status_venda       50000 non-null  str    
dtypes: float64(3), int64(3), str(9)
memory usage: 5.

None


--- Resumo Estatístico de Variáveis Numéricas ---


,id_item_venda,id_pedido,quantidade,preco_unitario,desconto,valor_total
count,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000
mean,24927.515980,24268.761320,1.485260,1849.68727,172.894776,2597.498904
std,14389.854334,8235.726963,1.213792,2158.36151,698.760053,4575.844584
min,1.000000,10000.000000,-5.000000,45.00000,0.000000,-21000.000000
25%,12467.750000,17158.750000,1.000000,299.90000,0.000000,414.575000
50%,24927.500000,24241.500000,1.000000,999.00000,0.000000,1284.115000
75%,37385.250000,31401.250000,2.000000,2800.00000,101.872500,2997.000000
max,49850.000000,38536.000000,10.000000,8500.00000,42000.000000,85000.000000



--- Primeiras 5 linhas do DataFrame ---


,id_item_venda,id_pedido,data_venda,nome_produto,categoria_produto,quantidade,preco_unitario,desconto,valor_total,nome_cliente,cidade_cliente,estado_cliente,nome_vendedor,forma_pagamento,status_venda
0,1,10000,2021-08-17 00:00:00,Mouse Gamer 10k DPI,Informática,1,250.0,0.00,250.00,Rafael Pereira,Caxias do Sul,RS,Larissa Mendes,Cartão de Crédito,Cancelado
1,2,10000,2021-08-17 00:00:00,Smartphone Premium X,Eletrônicos,1,4500.0,0.00,4500.00,Rafael Pereira,Caxias do Sul,RS,Larissa Mendes,Cartão de Crédito,Cancelado
2,3,10001,2021-02-24 00:00:00,"Smart TV 55"" 4K",Eletrônicos,1,3100.0,587.85,2512.15,Thiago Mendes,Anápolis,GO,Diana Souza,Cartão de Débito,Concluído
3,4,10002,2022-11-28 00:00:00,"Monitor 27"" 4K",Informática,1,2100.0,0.00,2100.00,Diego Costa,Rio de Janeiro,RJ,Gabriel Gomes,Cartão de Crédito,Concluído
4,5,10002,2022-11-28 00:00:00,Teclado Mecânico RGB,Informática,1,550.0,69.14,480.86,Diego Costa,Rio de Janeiro,RJ,Gabriel Gomes,Cartão de Crédito,Concluído


In [5]:
# 1. Quantidade de registros e colunas
linhas, colunas = df.shape
print(f"Quantidade de registros (linhas): {linhas}")
print(f"Quantidade de variáveis (colunas): {colunas}")

# 2. Verificação de valores nulos
print("\n--- Valores Nulos por Coluna ---")
nulos = df.isnull().sum()
display(nulos[nulos > 0] if nulos.sum() > 0 else "Nenhum valor nulo encontrado!")

# 3. Anomalias numéricas encontradas (valores negativos em quantidade e total)
print("\n--- Inconsistências de Valores Negativos ---")
print(f"Registros com quantidade negativa: {len(df[df['quantidade'] < 0])}")
print(f"Registros com valor total negativo: {len(df[df['valor_total'] < 0])}")

Quantidade de registros (linhas): 50000
Quantidade de variáveis (colunas): 15

--- Valores Nulos por Coluna ---


cidade_cliente    150
dtype: int64


--- Inconsistências de Valores Negativos ---
Registros com quantidade negativa: 62
Registros com valor total negativo: 87


In [6]:
# Criando uma cópia para preservar os dados originais
df_clean = df.copy()

# 1. Removendo linhas com valores nulos na coluna 'cidade_cliente'
df_clean = df_clean.dropna(subset=['cidade_cliente'])

# 2. Filtrando inconsistências lógicas (valores maiores que zero)
df_clean = df_clean[(df_clean['quantidade'] > 0) & (df_clean['valor_total'] > 0)]

# 3. Verificando e removendo registros duplicados
duplicados_antes = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

# 4. Convertendo a coluna 'data_venda' para datetime (coagindo erros em NaT e removendo-os)
df_clean['data_venda'] = pd.to_datetime(df_clean['data_venda'], errors='coerce')
linhas_data_invalida = df_clean['data_venda'].isnull().sum()
df_clean = df_clean.dropna(subset=['data_venda'])

# Verificação final pós-limpeza
print("--- Resultados do Tratamento de Dados ---")
print(f"Linhas duplicadas removidas: {duplicados_antes}")
print(f"Linhas com datas corrompidas removidas: {linhas_data_invalida}")
print(f"Registros restantes na base tratada: {df_clean.shape[0]} (Redução de {df.shape[0] - df_clean.shape[0]} linhas no total)")
print(f"Valores nulos restantes: {df_clean.isnull().sum().sum()}")
print(f"Menor quantidade na base: {df_clean['quantidade'].min()}")
print(f"Menor valor total na base: {df_clean['valor_total'].min()}")
print(f"Tipo de dado da coluna data_venda: {df_clean['data_venda'].dtype}")

--- Resultados do Tratamento de Dados ---
Linhas duplicadas removidas: 150
Linhas com datas corrompidas removidas: 149
Registros restantes na base tratada: 49402 (Redução de 598 linhas no total)
Valores nulos restantes: 0
Menor quantidade na base: 1
Menor valor total na base: 33.76
Tipo de dado da coluna data_venda: datetime64[us]


In [7]:

print("Status de venda disponíveis:", df_clean['status_venda'].unique())

# 2. Cálculo do Faturamento Bruto (soma do valor_total de todos os itens de pedidos não cancelados/gerais)
faturamento_bruto_total = df_clean['valor_total'].sum()

# 3. Cálculo do Faturamento Líquido
# Consideramos apenas vendas com status 'Concluído' (ou equivalente) para o faturamento real líquido
vendas_concluidas = df_clean[df_clean['status_venda'] == 'Concluído']
faturamento_liquido = vendas_concluidas['valor_total'].sum()

print("\n--- Resultados Financeiros ---")
print(f"Faturamento Bruto Total (Geral): R$ {faturamento_bruto_total:,.2f}")
print(f"Faturamento Líquido (Apenas Vendas Concluídas): R$ {faturamento_liquido:,.2f}")

Status de venda disponíveis: <StringArray>
['Cancelado', 'Concluído', 'Devolvido', 'Pendente']
Length: 4, dtype: str

--- Resultados Financeiros ---
Faturamento Bruto Total (Geral): R$ 128,858,233.36
Faturamento Líquido (Apenas Vendas Concluídas): R$ 109,961,957.56


In [ ]:
print("Lista de produtos mais vendidos", df_clean['nome_produto'])

In [8]:
# 1. Top 10 produtos por quantidade de transações
top_10_transacoes = df_clean['nome_produto'].value_counts().head(10).reset_index()
top_10_transacoes.columns = ['Nome do Produto', 'Frequência de Vendas']

# 2. Top 10 produtos por volume total de itens vendidos
top_10_quantidade = df_clean.groupby('nome_produto')['quantidade'].sum().sort_values(ascending=False).head(10).reset_index()
top_10_quantidade.columns = ['Nome do Produto', 'Total de Itens Vendidos']

print("--- TOP 10 PRODUTOS MAIS VENDIDOS (POR TRANSAÇÕES) ---")
display(top_10_transacoes)

print("\n--- TOP 10 PRODUTOS MAIS VENDIDOS (POR VOLUME DE ITENS) ---")
display(top_10_quantidade)

--- TOP 10 PRODUTOS MAIS VENDIDOS (POR TRANSAÇÕES) ---


,Nome do Produto,Frequência de Vendas
0,Teclado Mecânico RGB,3373
1,"Tablet 10""",3358
2,"Monitor 24"" Full HD",3334
3,"Smart TV 55"" 4K",3331
4,"Monitor 27"" 4K",3320
5,Caixa de Som Portátil,3316
6,Fone de Ouvido Bluetooth,3307
7,Smartwatch Fit Z,3290
8,Smartphone Básico Y,3284
9,Mouse Gamer 10k DPI,3283



--- TOP 10 PRODUTOS MAIS VENDIDOS (POR VOLUME DE ITENS) ---


,Nome do Produto,Total de Itens Vendidos
0,Teclado Mecânico RGB,5109
1,Smartphone Básico Y,5004
2,"Monitor 27"" 4K",4973
3,Smartphone Premium X,4930
4,Mouse Gamer 10k DPI,4923
5,"Smart TV 55"" 4K",4911
6,Notebook Estudante,4887
7,Cabo USB-C 2m,4883
8,"Tablet 10""",4882
9,Notebook Ultra Pro,4870
